# Part 1 · Notebook 02 — Macro regime dashboard

**Sessions:** S3 (central banks, rates, yield curve) · S4 (business cycle, dollar, intermarket) · **Clinic W1**

**You will build:** yield-curve and credit indicators with recession shading, market-implied Fed probabilities, the rolling stock–bond correlation, and a rule-based growth × inflation regime. Results are exported for your Excel dashboard.

> ⚠️ FRED values are today's **revised** values. That is fine for a dashboard of the present; for historical tests use vintages (notebook 01).

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # p1lib.py lives next to this notebook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p1lib as p

p.use_course_style()
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)   # CSV exports for Excel go here
pd.set_option("display.float_format", "{:,.4f}".format)
print("Offline fixtures:" if os.environ.get("P1_FIXTURES") else "Live data from FRED", os.environ.get("P1_FIXTURES", ""))

In [ ]:
START = "2000-01-01"          # ✏️ Change me
rec = p.fred_series("USREC", start=START)                       # NBER recession indicator (0/1)
curve = pd.concat({"10y - 2y": p.fred_series("T10Y2Y", start=START),
                   "10y - 3m": p.fred_series("T10Y3M", start=START)}, axis=1).dropna()
curve.tail()

## 1. Yield curve and recessions

In [ ]:
fig, ax = plt.subplots()
p.shade_periods(ax, rec.reindex(curve.index, method="ffill"))
curve.plot(ax=ax, title="Treasury yield-curve spreads (percentage points)")
ax.axhline(0, color="#52514e", lw=1)
ax.set_xlabel(""); ax.legend(loc="lower left")
plt.show()
inverted = (curve < 0).mean() * 100
print("Share of days inverted since", START, ":", inverted.round(1).to_dict(), "%")

## 2. What is the market pricing for the next FOMC meeting?

Fed funds futures settle on **100 − the average effective fed funds rate of the month**. ✏️ Enter today's futures price for the meeting month (from CME or your broker).

In [ ]:
FUTURES_PRICE = 95.73     # ✏️ Change me: fed funds futures price for the meeting month
CURRENT_RATE = 4.33       # ✏️ Change me: current effective fed funds rate (%)
MEETING_DAY = 17          # ✏️ Change me: day of the month of the FOMC decision
DAYS_IN_MONTH = 30        # ✏️ Change me

post = p.fed_funds_post_meeting_rate(FUTURES_PRICE, CURRENT_RATE, MEETING_DAY, DAYS_IN_MONTH)
prob = p.move_probability(post, CURRENT_RATE)
print(f"Implied rate after the meeting: {post:.3f}%")
print(f"Implied probability of a 25 bp {'hike' if prob > 0 else 'cut'}: {abs(prob):.0%}")

## 3. Credit spreads and the dollar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hy = p.fred_series("BAMLH0A0HYM2", start=START).dropna()
usd = p.fred_series("DTWEXBGS", start=START).dropna()
p.shade_periods(axes[0], rec.reindex(hy.index, method="ffill"))
hy.plot(ax=axes[0], title="High-yield credit spread (percentage points)", color=p.PALETTE[0])
usd.plot(ax=axes[1], title="Broad US dollar index", color=p.PALETTE[0])
for ax in axes: ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 4. Is the stock–bond correlation stable?

Daily S&P 500 returns vs an approximate 10-year Treasury return (−duration × yield change). FRED's S&P 500 series covers about the last 10 years.

In [ ]:
WINDOW = 252      # ✏️ Change me: rolling window in trading days
spx = p.fred_series("SP500").dropna()
y10 = p.fred_series("DGS10").dropna()
rets = pd.concat({"stocks": spx.pct_change(), "bonds": p.bond_return_from_yield(y10)}, axis=1).dropna()
roll_corr = rets["stocks"].rolling(WINDOW).corr(rets["bonds"]).dropna()
ax = roll_corr.plot(title=f"Rolling {WINDOW}-day correlation: S&P 500 vs 10-year Treasury")
ax.axhline(0, color="#52514e", lw=1); ax.set_xlabel(""); ax.set_ylim(-1, 1)
plt.show()
print("Latest correlation:", round(roll_corr.iloc[-1], 2))

## 5. Growth × inflation regime (explicit rules)

Rule: compare the **direction** of growth (industrial production YoY) and inflation (CPI YoY) over the last `LOOKBACK` months.

In [ ]:
LOOKBACK = 3      # ✏️ Change me
growth = p.yoy_pct(p.fred_series("INDPRO", start=START))
inflation = p.yoy_pct(p.fred_series("CPIAUCSL", start=START))
reg = p.regime(growth, inflation, lookback=LOOKBACK)
print("Current regime:", reg["regime"].iloc[-1], "as of", reg.index[-1].date())
reg[["growth", "inflation", "regime"]].tail(6)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
reg["growth"].plot(ax=axes[0], title="Growth: industrial production YoY (%)")
reg["inflation"].plot(ax=axes[1], title="Inflation: CPI YoY (%)", color=p.PALETTE[1])
for ax in axes:
    p.shade_periods(ax, rec.reindex(reg.index, method="ffill")); ax.axhline(0, color="#52514e", lw=1); ax.set_xlabel("")
plt.tight_layout(); plt.show()
(reg["regime"].value_counts(normalize=True) * 100).round(1).rename("% of months")

## 6. Dashboard summary → Excel

In [ ]:
summary = pd.Series({
    "10y-2y spread (pp)": curve["10y - 2y"].iloc[-1],
    "10y-3m spread (pp)": curve["10y - 3m"].iloc[-1],
    "HY credit spread (pp)": hy.iloc[-1],
    "Broad dollar index": usd.iloc[-1],
    "Stock-bond correlation (1y)": roll_corr.iloc[-1],
    "Implied post-FOMC rate (%)": post,
    "Regime": reg["regime"].iloc[-1],
}, name="value")
summary.to_csv(OUT_DIR / "macro_dashboard_summary.csv")
reg.to_csv(OUT_DIR / "macro_regime_history.csv")
p.log_research({"notebook": "02_macro_dashboard", "source": "FRED", "note": "latest (revised) values"})
summary

## Questions
1. How many months before past recessions did the curve invert? Is inversion a timing tool?
2. Why did the stock–bond correlation change sign in 2022?
3. Write the macro section of your report: current regime, key risks, and what would change your view.